# Ollama Raw LLM Completion


## Setup

Run Ollama before executing cells:
- `ollama serve`
- `ollama pull gemma3:4b` (or another <=12B local model)

## Ollama API

In [1]:
import requests
from fastapi import FastAPI
from pydantic import BaseModel
from fastapi.responses import StreamingResponse
import json
import uvicorn
import threading
from Modules.conversion_1.conversion import messages_to_prompt
from Modules.conversion_1.prompt_factor import PromptBuilder

# API Setting
OLLAMA_URL = "http://localhost:11434/api/generate"
MODEL = "gemma3:4b"

app = FastAPI(title="HKBU Study Companion")
builder = PromptBuilder()

class Query(BaseModel):
    """HKBU Study Assistant providing three mutually exclusive working modes.
    
    ====================   ===========   ===============   =====================
    Mode                    is_search     is_study_plan     Optional Features
    ====================   ===========   ===============   =====================
    Normal Mode             False         False             think_mode
    Search Mode (RAG)       True          False             use_embedding_retrieval
    Study Plan Mode         False         True              -
    ====================   ===========   ===============   =====================
    
    Constraints:
        - is_search and is_study_plan cannot both be True
        - use_embedding_retrieval only takes effect when is_search=True
        - think_mode only takes effect when is_search=False and is_study_plan=False
    """

    question: str                           # user query
    context: list = []                      # history record
    system_prompt: str = ""                 # system role setting
    is_search: bool = False                 # enable search mode?
    use_embedding_retrieval: bool = False   # using embedding retrieval?(if not, use lexical retrieval)
    think_mode: bool = False                # enable think mode?
    is_study_plan: bool = False             # ebable study plan mode?

def complete_document(prompt: str):
    """Generate streaming response from Ollama"""
    payload = {
        "model": MODEL,
        "prompt": prompt,
        "stream": True,
        "options": {
            "num_predict": 2048,
            "temperature": 0.3
        }
    }
    
    try:
        with requests.post(OLLAMA_URL, json=payload, stream=True, timeout=120) as r:
            r.raise_for_status()
            for line in r.iter_lines(decode_unicode=True):
                if not line:
                    continue
                    
                try:
                    data = json.loads(line)
                    token = data.get("response", "")
                    if token:
                        yield token
                    
                    if data.get("done", False):
                        break
                        
                except json.JSONDecodeError:
                    print(f"Failed to parse JSON: {line}")
                    continue
                    
    except requests.exceptions.ConnectionError:
        yield "\nError: Cannot connect to Ollama server. Please ensure Ollama is running."
    except requests.exceptions.Timeout:
        yield "\nError: Request timeout after 120 seconds."
    except Exception as e:
        yield f"\nError: {str(e)}"

# API
@app.post("/ask")
def ask(q: Query):
    
    full_prompt_list = builder.get_full_prompt_list(
        question=q.question,
        context = q.context,
        system_prompt = q.system_prompt,
        is_search = q.is_search,
        use_embedding_retrieval= q.use_embedding_retrieval,
        think_mode = q.think_mode,
        is_study_plan = q.is_study_plan
    )

    full_prompt = messages_to_prompt(full_prompt_list)
    return StreamingResponse(
        complete_document(full_prompt),
        media_type="text/plain; charset=utf-8"
    )

d:\env\anaconda\envs\comp7125-gp\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



Processing documents...
  Found existing index file, loading...
  Loaded 236 chunks
  BM25 index built (236 chunks)
Loading embedding model (first time will download ~100MB)...


Loading weights: 100%|██████████| 71/71 [00:00<00:00, 8559.80it/s]
BertModel LOAD REPORT from: BAAI/bge-small-zh-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Model loaded successfully
Generating embeddings for 236 chunks (may take 1-2 minutes)...


Batches: 100%|██████████| 8/8 [00:02<00:00,  3.30it/s]

✅ Embeddings generated, shape: (236, 512)

 RAG system initialized successfully!


In [ ]:
def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8326, log_level="info")
    print("✅ Backend Address: http://localhost:8326")
    print("✅ Backend Document: http://localhost:8326/docs")

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()

## Test

Here are the testing functions for each function of the model

In [4]:
from textwrap import dedent

def test_ask_directly(test_cases):
    for i, test_case in enumerate(test_cases):
        print(f"\n{'='*60}")
        print(f"test {i+1}: {test_case.get('description', '')}")
        print(f"question: {test_case.get('question', '')}")
        print(f"{'='*60}")
        
        query = Query(**test_case)
        
        full_prompt_list = builder.get_full_prompt_list(
            question=query.question,
            context=query.context,
            system_prompt=query.system_prompt,
            is_search=query.is_search,
            use_embedding_retrieval=query.use_embedding_retrieval,
            think_mode=query.think_mode,
            is_study_plan=query.is_study_plan
        )

        
        full_prompt = messages_to_prompt(full_prompt_list)
        print(f"full prompt: \n{full_prompt}")
        print(f"{'='*60}\n")
        
        print("answer: ", end="")
        for token in complete_document(full_prompt):
            print(token, end="", flush=True)
        print("\n")

# test cases
TEST_CASES = [
    {
        "description": "Normal Mode(no thinking)",
        "question": "What's mechine learning?",
    },
    {
        "description": "Normal Mode(enable thinking)",
        "question": dedent("""
            A train travels from City A to City B at 60 km/h. 
            On the return trip from City B to City A, it travels at 40 km/h. 
            What is the average speed for the entire round trip? 
            Please solve step by step.
        """).strip(),
        "think_mode": True,
    },
    {
        "description": "Search Mode(lexical retrieval)",
        "question": "What are the assessment methods and weighting for the course COMP7025?",
        "is_search": True,
    },
    {
        "description": "Search Mode(embedding retrieval)",
        "question": "What are the assessment methods and weighting for the course COMP7025?",
        "is_search": True,
        "use_embedding_retrieval": True,
    },
    {
        "description": "Search Mode(lexical retrieval)",
        "question": "What are the consequences of submitting homework late?",
        "is_search": True,
    },
    {
        "description": "Search Mode(embedding retrieval)",
        "question": "What are the consequences of submitting homework late?",
        "is_search": True,
        "use_embedding_retrieval": True,
    },
    {
        "description": "Study plan Mode",
        "question": dedent("""
        Please help me create a 14 day review plan. 
        I am only available in the evenings on weekdays, 
        but on weekends, I am available in the afternoons and evenings. 
        This plan starts from Monday
    """).strip(),
        "is_study_plan": True
    }
]

# test
test_ask_directly(TEST_CASES)


test 1: Normal Mode(no thinking)
question: What's mechine learning?
full prompt: 
System: 
User: Question: What's mechine learning?

Assistant: 

answer: Okay, here's an explanation of machine learning:

**Machine learning is a field of computer science that gives computers the ability to learn from data without being explicitly programmed.** 

Here's a breakdown of what that means:

* **Traditional Programming vs. Machine Learning:**  In traditional programming, you write specific instructions for a computer to follow to solve a problem.  If the situation changes, you have to rewrite the code. Machine learning is different. Instead of telling the computer *how* to do something, you feed it a lot of data and let it figure out the rules and patterns itself.

* **How it Works:**
    * **Data:** Machine learning algorithms need data to learn from. This data can be anything – images, text, numbers, sounds, etc.
    * **Algorithms:** These are the "learning" methods.  There are many differ

If you get `ConnectionError`, start Ollama first (`ollama serve`) and ensure your model is available (`ollama pull llama3.1:8b`).
If `requests` is missing, run `%pip install requests` in a notebook cell.
